In [165]:
import pandas as pd
import os

In [166]:
DEST_FILE = "../data"
FILE_NAME = "TMDB_all_movies.csv"
full_path = os.path.join(DEST_FILE, FILE_NAME)

In [167]:
# Lecture du CSV
df = pd.read_csv(full_path)

In [168]:
# Tri
df_sorted = df.sort_values(
    by=["vote_count", "popularity", "vote_average"],
    ascending=[False, False, False]
)

In [169]:
# Garder seulement les 50000 premiers
df = df_sorted.head(50000).copy()

In [170]:
# Conversions numériques
df["vote_average"] = pd.to_numeric(df["vote_average"], errors="coerce").astype(float)
df["vote_count"] = pd.to_numeric(df["vote_count"], errors="coerce").astype("Int64")  # Int64 pour accepter NaN
df["runtime"] = pd.to_numeric(df["runtime"], errors="coerce").astype(float)
df["budget"] = pd.to_numeric(df["budget"], errors="coerce").astype(float)
df["popularity"] = pd.to_numeric(df["popularity"], errors="coerce").astype(float)

In [171]:
# Dates
df["release_date"] = pd.to_datetime(df["release_date"], errors="coerce")
df["release_year"] = df["release_date"].dt.year.astype("Int64")

In [172]:
# Colonnes à transformer en listes
array_columns = [
    "genres", "production_countries", "production_companies", "cast", "director", "writers"
]

for col in array_columns:
    df[col + "_array"] = (
        df[col].str.split(r",\s*")       # découper sur virgule + espace
             .apply(lambda x: x if isinstance(x, list) else [])  # remplacer NaN par []
    )

# Suppression des colonnes originales
df = df.drop(columns=array_columns)

In [173]:
# Vérifier les types
print(df.dtypes)

id                                     int64
title                                 object
vote_average                         float64
vote_count                             Int64
status                                object
release_date                  datetime64[ns]
revenue                              float64
runtime                              float64
budget                               float64
imdb_id                               object
original_language                     object
original_title                        object
overview                              object
popularity                           float64
tagline                               object
spoken_languages                      object
director_of_photography               object
producers                             object
music_composer                        object
imdb_rating                          float64
imdb_votes                           float64
poster_path                           object
release_ye

In [174]:
df = df.drop(
    ["status", "imdb_id", "tagline", "director_of_photography",
     "producers", "imdb_rating", "imdb_votes",
     "music_composer", "revenue", "spoken_languages", "original_language"],
    axis=1
)

In [175]:
# Delete line with empty title
df = df[df["title"].notna() & (df["title"] != "")]

In [176]:
# Delete line with empty overview
df = df[df["overview"].notna() & (df["overview"] != "")]

In [177]:
df = df.fillna({
    'release_year': -1
})

In [178]:
df.shape

(49783, 18)

In [179]:
def count_empty_values(df):
    counts = {}
    for col in df.columns:
        counts[col] = (
            df[col].isna()                                  # NaN / None
            | (df[col] == "")                               # chaîne vide
            | (df[col].apply(lambda x: isinstance(x, list) and len(x) == 0))  # liste vide
        ).sum()
    return pd.Series(counts, name="empty_count")

In [180]:
empty_counts = count_empty_values(df)
print(empty_counts)

id                               0
title                            0
vote_average                     0
vote_count                       0
release_date                    10
runtime                          0
budget                           0
original_title                   0
overview                         0
popularity                       0
poster_path                    136
release_year                     0
genres_array                    77
production_countries_array     755
production_companies_array    2370
cast_array                     620
director_array                 151
writers_array                 2423
Name: empty_count, dtype: int64


In [181]:
clean_path = os.path.join(DEST_FILE, "TMDB_clean.csv")

In [182]:
if os.path.exists(clean_path):
    os.remove(clean_path)

In [183]:
df.to_csv(clean_path, index=False)